<a href="https://colab.research.google.com/github/KBCoronado/MachineLearning/blob/main/Practica_6_Suport_Vector_Machine_(SVM)_Prestamos_Lending_Club.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

In [4]:
prestamos_df = pd.read_csv('lending_club_2007_2011_6_states (2).csv')
prestamos_df.head()

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,hardship_flag,disbursement_method,debt_settlement_flag,debt_settlement_flag_date
0,2400,2400,2400.0,36 months,15.96,84.33,C,C5,NaN,10+ years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
1,10000,10000,10000.0,36 months,13.49,339.31,C,C1,AIR RESOURCES BOARD,10+ years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
2,3000,3000,3000.0,36 months,18.64,109.43,E,E1,MKC Accounting,9 years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
3,5600,5600,5600.0,60 months,21.28,152.39,F,F2,NaN,4 years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
4,5375,5375,5350.0,60 months,12.69,121.45,B,B5,Starbucks,< 1 year,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN


In [5]:
prestamos_df["grade_code"] = prestamos_df["grade"].astype("category").cat.codes
prestamos_df["purpose_code"] = prestamos_df["purpose"].astype("category").cat.codes
prestamos_df["addr_state_code"] = prestamos_df["addr_state"].astype("category").cat.codes
prestamos_df["home_ownership_code"] = prestamos_df["home_ownership"].astype("category").cat.codes

prestamos_df["repaid"] = prestamos_df["loan_status"].apply(
    lambda x: 1 if x == "Fully Paid" else 0
)

In [6]:
X = prestamos_df[['funded_amnt', 'int_rate', 'grade_code',
                  'purpose_code', 'addr_state_code',
                  'home_ownership_code', 'annual_inc',
                  'dti', 'revol_util',
                  'pub_rec_bankruptcies']].fillna(0)
y = prestamos_df["repaid"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.4, stratify=y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((11944, 10), (7964, 10), (11944,), (7964,))

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
svm_model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    class_weight='balanced',
    probability=True,
    random_state=42
)
svm_model.fit(X_train_scaled, y_train)

SVC(class_weight='balanced', probability=True, random_state=42)

In [10]:
y_pred_svm = svm_model.predict(X_test_scaled)
print("Reporte de Clasificación (SVM):")
print(classification_report(y_test, y_pred_svm, target_names=["No Pagado", "Pagado"]))
print("Matriz de Confusión (SVM):")
print(confusion_matrix(y_test, y_pred_svm))

Reporte de Clasificación (SVM):
              precision    recall  f1-score   support

   No Pagado       0.22      0.64      0.33      1177
      Pagado       0.91      0.62      0.73      6787

    accuracy                           0.62      7964
   macro avg       0.57      0.63      0.53      7964
weighted avg       0.81      0.62      0.67      7964

Matriz de Confusión (SVM):
[[ 753  424]
 [2611 4176]]


In [11]:
from sklearn.metrics import roc_curve, roc_auc_score
import plotly.graph_objects as go
y_prob_svm = svm_model.predict_proba(X_test_scaled)[:, 1]
auc = roc_auc_score(y_test, y_prob_svm)
print(f"\n AUC-ROC:{auc:.3f}")
fpr, tpr, thresholds = roc_curve(y_test, y_prob_svm)
fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name='SVM', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Aleatorio', line=dict(dash='dash')))
fig.update_layout(
    title=f'Probabilidades de la clase positiva - Curva ROC (AUC ={auc:.3f})',
    xaxis_title='Tasa de Falsos Positivos (1 - Especificidad)',
    yaxis_title='Tasa de Verdaderos Positivos (Sensibilidad)',
    width=700, height=500
)
fig.show()


 AUC-ROC:0.671


In [12]:
y_prob_svm = svm_model.predict_proba(X_test_scaled)[:, 0]
auc = roc_auc_score(y_test, y_prob_svm)
print(f"\nAUC-ROC: {auc:.3f}")
fpr, tpr, thresholds = roc_curve(y_test, y_prob_svm)
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=fpr,
        y=tpr,
        mode='lines',
        name='SVM',
        line=dict(color='blue')
    )
)
fig.add_trace(
    go.Scatter(
        x=[0,1],
        y=[0,1],
        mode='lines',
        name='Aleatorio',
        line=dict(dash='dash')
    )
)
fig.update_layout(
    title=f'Probabilidades de la clase negativa - Curva ROC (AUC = {auc:.3f})',
    xaxis_title='Tasa de Falsos Positivos (1 - Especificidad)',
    yaxis_title='Tasa de Verdaderos Positivos (Sensibilidad)',
    width=700,
    height=500
)
fig.show()


AUC-ROC: 0.329


In [13]:
from sklearn.metrics import roc_curve, roc_auc_score
import plotly.graph_objects as go
y_prob = svm_model.predict_proba(X_test_scaled)
fpr_pos, tpr_pos, _ = roc_curve(y_test, y_prob[:, 1], pos_label=1)
auc_pos = roc_auc_score(y_test, y_prob[:, 1])
fpr_neg, tpr_neg, _ = roc_curve(y_test, y_prob[:, 0], pos_label=0)
auc_neg = roc_auc_score(y_test, y_prob[:, 0])
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fpr_pos, y=tpr_pos,
    mode='lines',
    name=f'Clase positiva (Pagado) - AUC = {auc_pos:.3f}',
    line=dict(color='blue', width=2)
  ))
fig.add_trace(go.Scatter(
    x=fpr_neg, y=tpr_neg,
    mode='lines',
    name=f'Clase negativa (No Pagado) - AUC = {auc_neg:.3f}',
    line=dict(color='green', width=2)
  ))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    name='Aleatorio',
    line=dict(color='red', width=1, dash='dash')
  ))
fig.update_layout(
    title="Comparación de Curvas ROC - Clases Positiva y Negativa",
    xaxis_title="Tasa de Falsos Positivos (1 - Especificidad)",
    yaxis_title="Tasa de Verdaderos Positivos (Sensibilidad)",
    width=900, height=600,
    legend=dict(x=0.6, y=0.05)
 )
fig.show()

In [14]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, fbeta_score, classification_report, confusion_matrix
import numpy as np
f2_scorer = make_scorer(fbeta_score, beta=2, pos_label=0)
svm = SVC(probability=False, class_weight='balanced', random_state=42)
param_grid = {
    'C': [0.1,1,10],
    'kernel': ['linear','rbf','poly'],
    'gamma': ['scale','auto'],
    'degree': [2,3],# solo afecta al kernel 'poly'
  }
grid_search = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    scoring=f2_scorer,# optimizamos el F2-score
    cv=5,# validación cruzada de 5 particiones
    verbose=2,
    n_jobs=-1# usa todos los núcleos disponibles
  )
print("Entrenando GridSearchCV para SVM...\n")
grid_search.fit(X_train_scaled, y_train)
print("\n Búsqueda finalizada.")
print("Mejores hiperparámetros encontrados:")
print(grid_search.best_params_)
print("\nMejor puntaje promedio (F2-score):")
print(round(grid_search.best_score_,4))
svm_optimo = grid_search.best_estimator_
y_pred_opt = svm_optimo.predict(X_test_scaled)
print("\n Reporte de Clasificación (SVM optimizado paraimpagos):\n")
print(classification_report(y_test, y_pred_opt, target_names=["No Pagado","Pagado"]))
print("Matriz de Confusión:")
print(confusion_matrix(y_test, y_pred_opt))

Entrenando GridSearchCV para SVM...

Fitting 5 folds for each of 36 candidates, totalling 180 fits

 Búsqueda finalizada.
Mejores hiperparámetros encontrados:
{'C': 0.1, 'degree': 2, 'gamma': 'scale', 'kernel': 'rbf'}

Mejor puntaje promedio (F2-score):
0.4693

 Reporte de Clasificación (SVM optimizado paraimpagos):

              precision    recall  f1-score   support

   No Pagado       0.22      0.67      0.33      1177
      Pagado       0.91      0.58      0.71      6787

    accuracy                           0.59      7964
   macro avg       0.56      0.62      0.52      7964
weighted avg       0.81      0.59      0.65      7964

Matriz de Confusión:
[[ 784  393]
 [2851 3936]]
